In [ ]:

#  [CELL 1] LIBRARY SETUP
#  Run this once to install tools and import libraries.

import sys
import subprocess
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Install pdfplumber if missing (Standard for Colab)
try:
    import pdfplumber
except ImportError:
    print("Installing pdfplumber")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber"])
    import pdfplumber

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.optimize import minimize

# Set plots to look nice
sns.set_theme(style="whitegrid")
print("Libraries installed and imported.")

In [ ]:
#  [CELL 2] DATA EXTRACTION (SMART SCAN)

pdf_path = "MSR_thesis.pdf"
all_rows = []

if os.path.exists(pdf_path):
    print(f"Reading {pdf_path}")
    with pdfplumber.open(pdf_path) as pdf:
        # Pages 89-101 contain the data tables
        for i in range(88, 101):
            try:
                page = pdf.pages[i]
                tables = page.extract_tables()

                for table in tables:
                    for row in table:
                        # CLEANING LOGIC:
                        # 1. Filter out 'None' and empty strings ''
                        # 2. Keep only the real values
                        cleaned_row = [x for x in row if x is not None and str(x).strip() != ""]

                        # 3. Validation: A valid data row must have exactly 7 values
                        # (X, Z, Alpha, SCF, SFF, Obtained, Simulated)
                        if len(cleaned_row) == 7:
                            try:
                                # 4. Attempt to convert to float (Filters out headers like 'X_coord')
                                float_row = [float(x) for x in cleaned_row]
                                all_rows.append(float_row)
                            except ValueError:
                                continue # Skip header rows

            except IndexError:
                pass

    if all_rows:
        df_final = pd.DataFrame(all_rows, columns=[
            'X_coordinate', 'Z_coordinate', 'angle_of_attack',
            'SCF', 'SFF', 'SCF_SFF_Obtained', 'SCF_SFF_Simulated'
        ])

        # Remove any potential duplicates
        df_final.drop_duplicates(inplace=True)

        print(f"Recovered {len(df_final)} valid data rows.")
        print(df_final.head())
    else:
        print("No valid data found. Check PDF structure.")
else:
    print(f"File '{pdf_path}' not found. Please upload it to Colab.")

In [ ]:

#  [CELL 3] ANN SURROGATE MODELING (Data Augmentation)
#  Run this once. It trains the ANN and generates 'df_1M'.

if 'df_final' in locals() and not df_final.empty:
    print("\nTraining ANN Surrogate Model")

    # Prepare Data
    X_raw = df_final[['X_coordinate', 'Z_coordinate', 'angle_of_attack']]
    y_raw = df_final[['SCF', 'SFF']]

    # Scale Data (Crucial for ANN)
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_scaled = scaler_X.fit_transform(X_raw)
    y_scaled = scaler_y.fit_transform(y_raw)

    # Train ANN
    ann_model = MLPRegressor(hidden_layer_sizes=(64, 64),
                             activation='relu', solver='adam',
                             max_iter=5000, random_state=42)
    ann_model.fit(X_scaled, y_scaled)
    print(f"   ANN Validation Score (R²): {ann_model.score(X_scaled, y_scaled):.4f}")

    # Generate 1 Million Synthetic Rows
    print("Generating 1,000,000 synthetic design points")
    n_samples = 1_000_000

    syn_X = np.random.uniform(X_raw['X_coordinate'].min(), X_raw['X_coordinate'].max(), n_samples)
    syn_Z = np.random.uniform(X_raw['Z_coordinate'].min(), X_raw['Z_coordinate'].max(), n_samples)
    syn_Alpha = np.random.uniform(X_raw['angle_of_attack'].min(), X_raw['angle_of_attack'].max(), n_samples)

    syn_inputs = pd.DataFrame({'X_coordinate': syn_X, 'Z_coordinate': syn_Z, 'angle_of_attack': syn_Alpha})

    syn_inputs_scaled = scaler_X.transform(syn_inputs)
    syn_outputs_scaled = ann_model.predict(syn_inputs_scaled)
    syn_outputs = scaler_y.inverse_transform(syn_outputs_scaled)

    df_1M = syn_inputs.copy()
    df_1M['SCF'] = syn_outputs[:, 0]
    df_1M['SFF'] = syn_outputs[:, 1]
    df_1M['Efficiency'] = df_1M['SCF'] / (df_1M['SFF'] + 1e-6)

    print("1 Million Row Dataset ('df_1M') Created")
else:
    print("Data extraction failed in previous step.")

In [ ]:

#  [CELL 4] TRAIN RANDOM FOREST (The Slow Part)
#  Run this once. It creates 'rf_final'.

if 'df_1M' in locals():
    print("\nTraining Random Forest on 1,000,000 rows")
    print("(This uses 80% for training and 20% for testing)")

    X_big = df_1M[['X_coordinate', 'Z_coordinate', 'angle_of_attack']]
    y_big = df_1M['Efficiency']

    X_train, X_test, y_train, y_test = train_test_split(X_big, y_big, test_size=0.2, random_state=42)

    # n_jobs=-1 uses all processors to speed this up
    rf_final = RandomForestRegressor(n_estimators=50, max_depth=25, n_jobs=-1, random_state=42)
    rf_final.fit(X_train, y_train)

    # Evaluate immediately
    y_pred = rf_final.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("    Random Forest Trained:")
    print(f"   Test Set R² Score: {r2:.5f}")
    print(f"   Test Set MAE: {mae:.5f}")
else:
    print("Big dataset 'df_1M' not found. Run Cell 3.")

In [ ]:

#  [CELL 5] OPTIMIZATION (Finding Optimal Location)

if 'rf_final' in locals():
    print("Running Optimization Algorithm")

    def objective_function(inputs):
        # Wrap input in DataFrame to avoid feature name warnings
        input_df = pd.DataFrame([inputs], columns=['X_coordinate', 'Z_coordinate', 'angle_of_attack'])
        predicted_efficiency = rf_final.predict(input_df)
        return -predicted_efficiency[0] # Negative because we want to MAXIMIZE

    bounds = [
        (df_final['X_coordinate'].min(), df_final['X_coordinate'].max()),
        (df_final['Z_coordinate'].min(), df_final['Z_coordinate'].max()),
        (df_final['angle_of_attack'].min(), df_final['angle_of_attack'].max())
    ]

    # Start search from the best point in the 1M grid
    best_idx_1M = df_1M['Efficiency'].idxmax()
    initial_guess = df_1M.iloc[best_idx_1M][['X_coordinate', 'Z_coordinate', 'angle_of_attack']].values

    result = minimize(objective_function, initial_guess, bounds=bounds, method='SLSQP')

    print("\nOPTIMIZATION RESULTS :")
    print(f"Optimal X Coordinate:    {result.x[0]:.4f}")
    print(f"Optimal Z Coordinate:    {result.x[1]:.4f}")
    print(f"Optimal Angle of Attack: {result.x[2]:.4f}")
    print(f"Max Predicted Efficiency:{(-result.fun):.4f}")
else:
    print("Model 'rf_final' not found. Run Cell 4.")

In [ ]:
#  [CELL 6] VISUALIZATION (Predicted vs Actual)

if 'y_test' in locals() and 'y_pred' in locals():
    print("Generating Plot")

    plt.figure(figsize=(10, 6))

    # Plotting a random subset of 1000 points to keep the plot clean and fast
    subset_idx = np.random.choice(len(y_test), 1000, replace=False)

    # Scatter plot of Actual vs Predicted
    plt.scatter(y_test.iloc[subset_idx], y_pred[subset_idx],
                alpha=0.3, color='blue', label='Data Points (Subset)')

    # Perfect fit line
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', label='Perfect Fit')

    plt.xlabel('Actual Efficiency (Surrogate)')
    plt.ylabel('Predicted Efficiency (Random Forest)')
    plt.title('Random Forest Performance: Predicted vs Actual')
    plt.legend()
    plt.show()
    print("Plot generated.")
else:
    print("Predictions not found. Run Cell 4.")

In [ ]:
# [CELL 7] 3D VISUALIZATION

from mpl_toolkits.mplot3d import Axes3D

print("Generating 3D Landscape of Efficiency")

# Create a grid of points to plot
x_range = np.linspace(df_final['X_coordinate'].min(), df_final['X_coordinate'].max(), 50)
z_range = np.linspace(df_final['Z_coordinate'].min(), df_final['Z_coordinate'].max(), 50)
X_grid, Z_grid = np.meshgrid(x_range, z_range)

# Fix Angle of Attack to the optimal value found
optimal_alpha = 30.0373 # Use the value from your results
inputs_grid = np.array([X_grid.ravel(), Z_grid.ravel(), np.full(X_grid.size, optimal_alpha)]).T

# Predict Efficiency for the grid
# Note: We use the ANN or RF here. RF is sharper.
input_df = pd.DataFrame(inputs_grid, columns=['X_coordinate', 'Z_coordinate', 'angle_of_attack'])
efficiency_pred = rf_final.predict(input_df).reshape(X_grid.shape)

# Plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X_grid, Z_grid, efficiency_pred, cmap='viridis', edgecolor='none')

ax.set_xlabel('X Coordinate')
ax.set_ylabel('Z Coordinate')
ax.set_zlabel('Efficiency (SCF/SFF)')
ax.set_title(f'Efficiency Landscape at Angle = {optimal_alpha:.1f}°')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()

In [ ]:
# [CELL 7] FEATURE IMPORTANCE RANKING


if 'rf_final' in locals():
    print("Calculating Feature Importance")

    # 1. Get numerical importance values
    # These sum up to 1.0 (100%)
    importances = rf_final.feature_importances_

    # 2. Map them to the names
    feature_names = ['X_coordinate', 'Z_coordinate', 'angle_of_attack']

    # Create a DataFrame for nice display
    feature_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("\nFeature Importance Ranking:")
    print(feature_df)

    # 3. Visualization
    plt.figure(figsize=(10, 6))

    # Create bar plot
    bars = plt.barh(feature_df['Feature'], feature_df['Importance'], color='#2c3e50')

    # Add percentage labels
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{width:.1%}', ha='left', va='center', fontsize=12)

    plt.xlabel('Importance Score (Impact on Efficiency)')
    plt.title('Which Design Parameter Matters Most?')
    plt.gca().invert_yaxis() # Put the most important at the top
    plt.show()

else:
    print("Model 'rf_final' not found. Please run Cell 4 first.")